In [53]:
import pandas as pd

In [54]:
df = pd.read_csv('processed_features.csv')
df=df[df['price_area']=='DK1']

In [55]:
df.head()

,datetime_utc,price_area,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,is_forecast,updated_at
0,2026-05-28 20:00:00+00,DK1,132.000000,2943.718720,8.6,19,False,2026-05-29 05:06:21.566775+00
2,2026-05-28 19:00:00+00,DK1,126.583333,1626.056421,9.4,98,False,2026-05-29 05:06:21.566775+00
4,2026-05-28 18:00:00+00,DK1,121.416667,1033.086950,10.4,210,False,2026-05-29 05:06:21.566775+00
6,2026-05-28 17:00:00+00,DK1,105.500000,756.656793,10.8,347,False,2026-05-29 05:06:21.566775+00
8,2026-05-28 16:00:00+00,DK1,89.500000,296.251229,12.6,486,False,2026-05-29 05:06:21.566775+00


In [56]:
# ============================================================
# STEP 3 — Clean current base dataframe
# Output: base_df
# ============================================================

# Change this line only if your dataframe has another name
base_df = df.copy()

# Clean column names just in case
base_df.columns = base_df.columns.str.strip()

# Convert datetime to UTC
base_df["datetime_utc"] = pd.to_datetime(base_df["datetime_utc"], utc=True)

# Keep DK1 only
base_df = base_df[base_df["price_area"] == "DK1"].copy()

# Sort by time
base_df = base_df.sort_values("datetime_utc")

# Drop duplicate timestamps if any
base_df = base_df.drop_duplicates(subset=["datetime_utc"], keep="last")

# Set datetime as index
base_df = base_df.set_index("datetime_utc")
base_df.index.name = "timestamp_utc"

# Remove columns we do not need for ML dataset
drop_cols = ["price_area", "is_forecast", "updated_at"]
base_df = base_df.drop(columns=[c for c in drop_cols if c in base_df.columns])

# Convert numeric columns
numeric_cols = [
    "co2_emissions_g_kwh",
    "spot_price_dkk_kwh",
    "wind_speed",
    "solar_radiation"
]

for col in numeric_cols:
    base_df[col] = pd.to_numeric(base_df[col], errors="coerce")

# Patch small CO2 missing gaps only
base_df["co2_emissions_g_kwh"] = (
    base_df["co2_emissions_g_kwh"]
    .interpolate(method="time", limit=3)
)

# Build expected hourly index
full_index = pd.date_range(
    base_df.index.min(),
    base_df.index.max(),
    freq="h",
    tz="UTC"
)

# Reindex to check missing hours
base_df = base_df.reindex(full_index)
base_df.index.name = "timestamp_utc"

print("✅ base_df created")
print("Shape:", base_df.shape)
print("From:", base_df.index.min())
print("To:", base_df.index.max())

print("\nMissing values:")
display(base_df.isna().sum().to_frame("missing_count"))

print("\nMissing percentage:")
display((base_df.isna().mean() * 100).round(4).to_frame("missing_%"))

print("\nDuplicate timestamps:", base_df.index.duplicated().sum())

display(base_df.head())
display(base_df.tail())

✅ base_df created
Shape: (47373, 4)
From: 2021-01-01 00:00:00+00:00
To: 2026-05-28 20:00:00+00:00

Missing values:


,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0



Missing percentage:


,missing_%
co2_emissions_g_kwh,0.0
spot_price_dkk_kwh,0.0
wind_speed,0.0
solar_radiation,0.0



Duplicate timestamps: 0


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation
timestamp_utc,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,5.2,0
2021-01-01 01:00:00+00:00,196.000000,0.33246,4.8,0
2021-01-01 02:00:00+00:00,168.166667,0.31937,6.5,0
2021-01-01 03:00:00+00:00,160.083333,0.30054,6.6,0
2021-01-01 04:00:00+00:00,161.083333,0.29913,5.6,0


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation
timestamp_utc,,,,
2026-05-28 16:00:00+00:00,89.500000,296.251229,12.6,486
2026-05-28 17:00:00+00:00,105.500000,756.656793,10.8,347
2026-05-28 18:00:00+00:00,121.416667,1033.086950,10.4,210
2026-05-28 19:00:00+00:00,126.583333,1626.056421,9.4,98
2026-05-28 20:00:00+00:00,132.000000,2943.718720,8.6,19


In [57]:
# ============================================================
# STEP 4 — Add temperature from Open-Meteo
# Input: base_df
# Output: base_df with temperature column
# Location: Aalborg, DK1
# ============================================================

import requests
import pandas as pd
import time
import numpy as np

# Aalborg coordinates
LATITUDE = 57.0488
LONGITUDE = 9.9217

start_date = base_df.index.min().date().strftime("%Y-%m-%d")
end_date = base_df.index.max().date().strftime("%Y-%m-%d")

print("🌡️ Fetching temperature from Open-Meteo")
print("Range:", start_date, "→", end_date)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": "temperature_2m",
    "timezone": "UTC"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

data = response.json()

if "hourly" not in data:
    raise ValueError(f"No hourly data returned: {data}")

temperature_df = pd.DataFrame({
    "timestamp_utc": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"]
})

temperature_df = (
    temperature_df
    .drop_duplicates(subset=["timestamp_utc"])
    .set_index("timestamp_utc")
    .sort_index()
)

temperature_df["temperature"] = pd.to_numeric(
    temperature_df["temperature"],
    errors="coerce"
)

# Align temperature to base_df hourly index
base_df = base_df.join(temperature_df, how="left")

print("\n✅ temperature added to base_df")
print("base_df shape:", base_df.shape)

print("\nMissing values:")
display(base_df.isna().sum().to_frame("missing_count"))

print("\nTemperature missing only:")
display(base_df[["temperature"]].isna().sum().to_frame("missing_count"))

display(base_df.head())
display(base_df.tail())

🌡️ Fetching temperature from Open-Meteo
Range: 2021-01-01 → 2026-05-28

✅ temperature added to base_df
base_df shape: (47373, 5)

Missing values:


,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0
temperature,0



Temperature missing only:


,missing_count
temperature,0


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature
timestamp_utc,,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,5.2,0,0.7
2021-01-01 01:00:00+00:00,196.000000,0.33246,4.8,0,1.0
2021-01-01 02:00:00+00:00,168.166667,0.31937,6.5,0,1.2
2021-01-01 03:00:00+00:00,160.083333,0.30054,6.6,0,1.3
2021-01-01 04:00:00+00:00,161.083333,0.29913,5.6,0,1.2


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature
timestamp_utc,,,,,
2026-05-28 16:00:00+00:00,89.500000,296.251229,12.6,486,18.8
2026-05-28 17:00:00+00:00,105.500000,756.656793,10.8,347,17.7
2026-05-28 18:00:00+00:00,121.416667,1033.086950,10.4,210,16.5
2026-05-28 19:00:00+00:00,126.583333,1626.056421,9.4,98,15.4
2026-05-28 20:00:00+00:00,132.000000,2943.718720,8.6,19,14.0


In [63]:
check_date = pd.Timestamp("2026-04-07 21:00:00", tz="UTC")

before = base_df.loc[base_df.index <= check_date, "spot_price_dkk_kwh"]
after = base_df.loc[base_df.index > check_date, "spot_price_dkk_kwh"]

print("Before / at 2026-04-08 21:00")
print(before.describe())

print("\nAfter 2026-04-08 21:00")
print(after.describe())

print("\nRows after this date that look like MWh:")
display(
    base_df.loc[
        (base_df.index > check_date) &
        (base_df["spot_price_dkk_kwh"].abs() > 20),
        ["spot_price_dkk_kwh"]
    ].head(20)
)

print("\nCount after this date that look like MWh:")
print((after.abs() > 20).sum())

Before / at 2026-04-08 21:00
count    46150.000000
mean         0.809092
std          0.715229
min         -3.277390
25%          0.414075
50%          0.665645
75%          0.945773
max          6.982420
Name: spot_price_dkk_kwh, dtype: float64

After 2026-04-08 21:00
count    1223.000000
mean      706.517026
std       386.492555
min      -233.736758
25%       488.589881
50%       807.843044
75%       952.451226
max      2943.718720
Name: spot_price_dkk_kwh, dtype: float64

Rows after this date that look like MWh:


,spot_price_dkk_kwh
timestamp_utc,
2026-04-07 23:00:00+00:00,859.138248
2026-04-08 00:00:00+00:00,800.859978
2026-04-08 01:00:00+00:00,763.794880
2026-04-08 02:00:00+00:00,753.407692
2026-04-08 03:00:00+00:00,773.416113
2026-04-08 04:00:00+00:00,803.419401
2026-04-08 05:00:00+00:00,873.775805
2026-04-08 06:00:00+00:00,1087.964967
2026-04-08 07:00:00+00:00,1261.389934



Count after this date that look like MWh:
1157


In [64]:
check_date = pd.Timestamp("2026-04-07 21:00:00", tz="UTC")

mask_after_mwh = base_df.index > check_date

base_df.loc[mask_after_mwh, "spot_price_dkk_kwh"] = (
    base_df.loc[mask_after_mwh, "spot_price_dkk_kwh"] / 1000
)

print("Fixed rows:", mask_after_mwh.sum())

print("\nAfter fixing:")
print(base_df["spot_price_dkk_kwh"].describe())

display(base_df.loc["2026-04-07 18:00:00":"2026-04-08 06:00:00", ["spot_price_dkk_kwh"]])

Fixed rows: 1223

After fixing:
count    47373.000000
mean         0.806444
std          0.708847
min         -3.277390
25%          0.415030
50%          0.669660
75%          0.946381
max          6.982420
Name: spot_price_dkk_kwh, dtype: float64


,spot_price_dkk_kwh
timestamp_utc,
2026-04-07 18:00:00+00:00,1.633264
2026-04-07 19:00:00+00:00,1.245217
2026-04-07 20:00:00+00:00,1.006919
2026-04-07 21:00:00+00:00,0.955509
2026-04-07 22:00:00+00:00,0.000864
2026-04-07 23:00:00+00:00,0.859138
2026-04-08 00:00:00+00:00,0.800860
2026-04-08 01:00:00+00:00,0.763795
2026-04-08 02:00:00+00:00,0.753408


In [66]:
base_df.tail()

,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature
timestamp_utc,,,,,
2026-05-28 16:00:00+00:00,89.500000,0.296251,12.6,486,18.8
2026-05-28 17:00:00+00:00,105.500000,0.756657,10.8,347,17.7
2026-05-28 18:00:00+00:00,121.416667,1.033087,10.4,210,16.5
2026-05-28 19:00:00+00:00,126.583333,1.626056,9.4,98,15.4
2026-05-28 20:00:00+00:00,132.000000,2.943719,8.6,19,14.0


In [67]:
# ============================================================
# Validate spot price after unit fix
# ============================================================

print("Missing values:")
display(base_df.isna().sum().to_frame("missing_count"))

print("\nSpot price summary:")
print(base_df["spot_price_dkk_kwh"].describe())

print("\nRows still looking like DKK/MWh:")
print((base_df["spot_price_dkk_kwh"].abs() > 20).sum())

print("\nHighest spot prices:")
display(base_df[["spot_price_dkk_kwh"]].sort_values("spot_price_dkk_kwh", ascending=False).head(10))

print("\nLowest spot prices:")
display(base_df[["spot_price_dkk_kwh"]].sort_values("spot_price_dkk_kwh").head(10))

Missing values:


,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0
temperature,0



Spot price summary:
count    47373.000000
mean         0.806444
std          0.708847
min         -3.277390
25%          0.415030
50%          0.669660
75%          0.946381
max          6.982420
Name: spot_price_dkk_kwh, dtype: float64

Rows still looking like DKK/MWh:
0

Highest spot prices:


,spot_price_dkk_kwh
timestamp_utc,
2024-12-12 16:00:00+00:00,6.98242
2022-08-29 17:00:00+00:00,6.47824
2022-08-29 18:00:00+00:00,6.40304
2022-08-30 17:00:00+00:00,6.36333
2022-08-24 17:00:00+00:00,6.32217
2022-08-29 16:00:00+00:00,6.23026
2024-12-12 15:00:00+00:00,6.10764
2022-08-26 06:00:00+00:00,5.97237
2022-08-29 19:00:00+00:00,5.95016



Lowest spot prices:


,spot_price_dkk_kwh
timestamp_utc,
2023-07-02 12:00:00+00:00,-3.27739
2023-07-02 13:00:00+00:00,-2.97133
2023-07-02 11:00:00+00:00,-1.98773
2023-07-02 10:00:00+00:00,-1.25079
2023-05-28 12:00:00+00:00,-0.96804
2023-05-28 11:00:00+00:00,-0.96767
2023-07-02 14:00:00+00:00,-0.92498
2023-05-29 12:00:00+00:00,-0.81527
2023-07-02 09:00:00+00:00,-0.73062


In [68]:
# ============================================================
# STEP 6 — Add time and calendar features
# Input: base_df
# Output: base_df with time features
# ============================================================

import numpy as np

try:
    import holidays
except ImportError:
    !pip install holidays
    import holidays

# Use Denmark local time for human/charging behavior
dk_time = base_df.index.tz_convert("Europe/Copenhagen")

base_df["hour"] = dk_time.hour
base_df["day_of_week"] = dk_time.dayofweek   # Monday=0, Sunday=6
base_df["month"] = dk_time.month
base_df["day_of_year"] = dk_time.dayofyear

# Cyclical time features
base_df["hour_sin"] = np.sin(2 * np.pi * base_df["hour"] / 24)
base_df["hour_cos"] = np.cos(2 * np.pi * base_df["hour"] / 24)

base_df["month_sin"] = np.sin(2 * np.pi * base_df["month"] / 12)
base_df["month_cos"] = np.cos(2 * np.pi * base_df["month"] / 12)

base_df["day_of_year_sin"] = np.sin(2 * np.pi * base_df["day_of_year"] / 366)
base_df["day_of_year_cos"] = np.cos(2 * np.pi * base_df["day_of_year"] / 366)

# Weekend
base_df["is_weekend"] = base_df["day_of_week"].isin([5, 6]).astype(int)

# Danish holidays
dk_holidays = holidays.country_holidays("DK")

base_df["is_holiday"] = [
    int(date in dk_holidays)
    for date in dk_time.date
]

print("✅ Time/calendar features added")
print("Shape:", base_df.shape)

print("\nMissing values:")
display(base_df.isna().sum().to_frame("missing_count"))

display(base_df.head())
display(base_df.tail())

✅ Time/calendar features added
Shape: (47373, 17)

Missing values:


,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0
temperature,0
hour,0
day_of_week,0
month,0
day_of_year,0
hour_sin,0


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,month_sin,month_cos,day_of_year_sin,day_of_year_cos,is_weekend,is_holiday
timestamp_utc,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,5.2,0,0.7,1,4,1,1,0.258819,0.965926,0.5,0.866025,0.017166,0.999853,0,1
2021-01-01 01:00:00+00:00,196.000000,0.33246,4.8,0,1.0,2,4,1,1,0.500000,0.866025,0.5,0.866025,0.017166,0.999853,0,1
2021-01-01 02:00:00+00:00,168.166667,0.31937,6.5,0,1.2,3,4,1,1,0.707107,0.707107,0.5,0.866025,0.017166,0.999853,0,1
2021-01-01 03:00:00+00:00,160.083333,0.30054,6.6,0,1.3,4,4,1,1,0.866025,0.500000,0.5,0.866025,0.017166,0.999853,0,1
2021-01-01 04:00:00+00:00,161.083333,0.29913,5.6,0,1.2,5,4,1,1,0.965926,0.258819,0.5,0.866025,0.017166,0.999853,0,1


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,month_sin,month_cos,day_of_year_sin,day_of_year_cos,is_weekend,is_holiday
timestamp_utc,,,,,,,,,,,,,,,,,
2026-05-28 16:00:00+00:00,89.500000,0.296251,12.6,486,18.8,18,3,5,148,-1.000000,-1.836970e-16,0.5,-0.866025,0.565345,-0.824855,0,0
2026-05-28 17:00:00+00:00,105.500000,0.756657,10.8,347,17.7,19,3,5,148,-0.965926,2.588190e-01,0.5,-0.866025,0.565345,-0.824855,0,0
2026-05-28 18:00:00+00:00,121.416667,1.033087,10.4,210,16.5,20,3,5,148,-0.866025,5.000000e-01,0.5,-0.866025,0.565345,-0.824855,0,0
2026-05-28 19:00:00+00:00,126.583333,1.626056,9.4,98,15.4,21,3,5,148,-0.707107,7.071068e-01,0.5,-0.866025,0.565345,-0.824855,0,0
2026-05-28 20:00:00+00:00,132.000000,2.943719,8.6,19,14.0,22,3,5,148,-0.500000,8.660254e-01,0.5,-0.866025,0.565345,-0.824855,0,0


In [69]:
# ============================================================
# STEP 7 — Add CO2 lag, rolling, and difference features
# Input: base_df
# Output: base_df with CO2 history features
# ============================================================

# Make sure data is sorted
base_df = base_df.sort_index()

target_col = "co2_emissions_g_kwh"

# Lag features
base_df["co2_lag_1h"] = base_df[target_col].shift(1)
base_df["co2_lag_2h"] = base_df[target_col].shift(2)
base_df["co2_lag_24h"] = base_df[target_col].shift(24)
base_df["co2_lag_168h"] = base_df[target_col].shift(168)

# Rolling features using only past values
past_co2 = base_df[target_col].shift(1)

base_df["co2_rolling_3h"] = past_co2.rolling(window=3).mean()
base_df["co2_rolling_6h"] = past_co2.rolling(window=6).mean()
base_df["co2_rolling_24h"] = past_co2.rolling(window=24).mean()

# Difference features using only past values
base_df["co2_diff_1h"] = base_df[target_col].shift(1) - base_df[target_col].shift(2)
base_df["co2_diff_24h"] = base_df[target_col].shift(1) - base_df[target_col].shift(25)

print("✅ CO2 lag/rolling/diff features added")
print("Shape:", base_df.shape)

print("\nMissing values:")
display(base_df.isna().sum().to_frame("missing_count"))

display(base_df.head(10))
display(base_df.tail())

✅ CO2 lag/rolling/diff features added
Shape: (47373, 26)

Missing values:


,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0
temperature,0
hour,0
day_of_week,0
month,0
day_of_year,0
hour_sin,0


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,...,is_holiday,co2_lag_1h,co2_lag_2h,co2_lag_24h,co2_lag_168h,co2_rolling_3h,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,5.2,0,0.7,1,4,1,1,0.258819,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-01 01:00:00+00:00,196.000000,0.33246,4.8,0,1.0,2,4,1,1,0.500000,...,1,190.833333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-01 02:00:00+00:00,168.166667,0.31937,6.5,0,1.2,3,4,1,1,0.707107,...,1,196.000000,190.833333,NaN,NaN,NaN,NaN,NaN,5.166667,NaN
2021-01-01 03:00:00+00:00,160.083333,0.30054,6.6,0,1.3,4,4,1,1,0.866025,...,1,168.166667,196.000000,NaN,NaN,185.000000,NaN,NaN,-27.833333,NaN
2021-01-01 04:00:00+00:00,161.083333,0.29913,5.6,0,1.2,5,4,1,1,0.965926,...,1,160.083333,168.166667,NaN,NaN,174.750000,NaN,NaN,-8.083333,NaN
2021-01-01 05:00:00+00:00,164.750000,0.29489,4.5,0,1.1,6,4,1,1,1.000000,...,1,161.083333,160.083333,NaN,NaN,163.111111,NaN,NaN,1.000000,NaN
2021-01-01 06:00:00+00:00,172.583333,0.29831,4.2,0,1.0,7,4,1,1,0.965926,...,1,164.750000,161.083333,NaN,NaN,161.972222,173.486111,NaN,3.666667,NaN
2021-01-01 07:00:00+00:00,177.416667,0.30709,2.9,0,0.9,8,4,1,1,0.866025,...,1,172.583333,164.750000,NaN,NaN,166.138889,170.444444,NaN,7.833333,NaN
2021-01-01 08:00:00+00:00,182.500000,0.33395,2.5,0,1.0,9,4,1,1,0.707107,...,1,177.416667,172.583333,NaN,NaN,171.583333,167.347222,NaN,4.833333,NaN


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,...,is_holiday,co2_lag_1h,co2_lag_2h,co2_lag_24h,co2_lag_168h,co2_rolling_3h,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2026-05-28 16:00:00+00:00,89.500000,0.296251,12.6,486,18.8,18,3,5,148,-1.000000,...,0,78.166667,83.250000,19.000000,85.750000,112.138889,150.222222,85.854167,-5.083333,63.833333
2026-05-28 17:00:00+00:00,105.500000,0.756657,10.8,347,17.7,19,3,5,148,-0.965926,...,0,89.500000,78.166667,28.750000,106.750000,83.638889,133.958333,88.791667,11.333333,70.500000
2026-05-28 18:00:00+00:00,121.416667,1.033087,10.4,210,16.5,20,3,5,148,-0.866025,...,0,105.500000,89.500000,35.333333,105.500000,91.055556,118.916667,91.989583,16.000000,76.750000
2026-05-28 19:00:00+00:00,126.583333,1.626056,9.4,98,15.4,21,3,5,148,-0.707107,...,0,121.416667,105.500000,51.833333,119.666667,105.472222,108.805556,95.576389,15.916667,86.083333
2026-05-28 20:00:00+00:00,132.000000,2.943719,8.6,19,14.0,22,3,5,148,-0.500000,...,0,126.583333,121.416667,50.333333,121.750000,117.833333,100.736111,98.690972,5.166667,74.750000


In [70]:
base_df.to_csv("base_df.csv", index=True)

In [71]:
# ============================================================
# STEP 8 — Add forecast wind + solar generation from EDS Forecasts_Hour
# Input: base_df
# Output: base_df with:
#   forecast_wind_generation_mw
#   forecast_solar_generation_mw
# ============================================================

import requests
import pandas as pd
import numpy as np
import time
import json
import calendar

start_utc = base_df.index.min()
end_utc = base_df.index.max()

print("🌬️☀️ Fetching DK1 forecast wind + solar generation")
print("Range:", start_utc, "→", end_utc)

def request_with_retry(url, params, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=60)

        if r.status_code == 200:
            return r

        if r.status_code == 429:
            sleep_time = 20 + attempt * 20
            print(f"⚠️ 429 rate limit. Sleeping {sleep_time} seconds then retrying...")
            time.sleep(sleep_time)
            continue

        print("⚠️ Request failed:", r.status_code)
        print(r.text[:300])
        return r

    return r


def fetch_forecasts_hour_dk1(start_utc, end_utc):
    all_rows = []

    url = "https://api.energidataservice.dk/dataset/Forecasts_Hour"

    for year in range(start_utc.year, end_utc.year + 1):
        for month in range(1, 13):

            month_start = pd.Timestamp(f"{year}-{month:02d}-01 00:00:00", tz="UTC")
            last_day = calendar.monthrange(year, month)[1]
            month_end = pd.Timestamp(f"{year}-{month:02d}-{last_day:02d} 23:00:00", tz="UTC")

            if month_end < start_utc or month_start > end_utc:
                continue

            api_start = max(month_start, start_utc)
            api_end = min(month_end, end_utc) + pd.Timedelta(hours=1)

            limit = 5000
            offset = 0

            while True:
                params = {
                    "start": api_start.strftime("%Y-%m-%dT%H:%M"),
                    "end": api_end.strftime("%Y-%m-%dT%H:%M"),
                    "filter": json.dumps({"PriceArea": ["DK1"]}),
                    "columns": "HourUTC,PriceArea,ForecastType,ForecastDayAhead",
                    "sort": "HourUTC",
                    "limit": limit,
                    "offset": offset
                }

                r = request_with_retry(url, params)

                if r.status_code != 200:
                    print(f"❌ Failed {year}-{month:02d}")
                    break

                records = r.json().get("records", [])

                if len(records) == 0:
                    break

                all_rows.extend(records)

                print(
                    f"Forecasts_Hour {year}-{month:02d}: "
                    f"{len(records)} rows | offset {offset}"
                )

                if len(records) < limit:
                    break

                offset += limit
                time.sleep(0.3)

            time.sleep(0.5)

    return pd.DataFrame(all_rows)


forecast_raw_df = fetch_forecasts_hour_dk1(start_utc, end_utc)

print("\nRaw forecast shape:", forecast_raw_df.shape)

if forecast_raw_df.empty:
    raise ValueError("No Forecasts_Hour data fetched.")

print("\nColumns:")
print(forecast_raw_df.columns.tolist())

print("\nForecast types found:")
display(forecast_raw_df["ForecastType"].value_counts().to_frame("count"))

# Clean
forecast_raw_df["timestamp_utc"] = pd.to_datetime(
    forecast_raw_df["HourUTC"],
    utc=True,
    errors="coerce"
)

forecast_raw_df["ForecastDayAhead"] = pd.to_numeric(
    forecast_raw_df["ForecastDayAhead"],
    errors="coerce"
)

forecast_raw_df["forecast_type_clean"] = (
    forecast_raw_df["ForecastType"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep only wind and solar
wind_mask = forecast_raw_df["forecast_type_clean"].str.contains("wind", na=False)
solar_mask = forecast_raw_df["forecast_type_clean"].str.contains("solar", na=False)

forecast_long_df = forecast_raw_df[wind_mask | solar_mask].copy()

forecast_long_df["feature"] = np.where(
    forecast_long_df["forecast_type_clean"].str.contains("solar", na=False),
    "forecast_solar_generation_mw",
    "forecast_wind_generation_mw"
)

# Sum wind types if onshore/offshore are separate
forecast_generation_df = (
    forecast_long_df
    .groupby(["timestamp_utc", "feature"], as_index=False)["ForecastDayAhead"]
    .sum()
    .pivot(index="timestamp_utc", columns="feature", values="ForecastDayAhead")
    .sort_index()
)

# Force same index as base_df
forecast_generation_df = forecast_generation_df.reindex(base_df.index)

# Make sure both columns exist
for col in ["forecast_wind_generation_mw", "forecast_solar_generation_mw"]:
    if col not in forecast_generation_df.columns:
        forecast_generation_df[col] = np.nan

forecast_generation_df = forecast_generation_df[
    ["forecast_wind_generation_mw", "forecast_solar_generation_mw"]
]

# Remove old columns if rerunning cell
base_df = base_df.drop(
    columns=[
        "forecast_wind_generation_mw",
        "forecast_solar_generation_mw"
    ],
    errors="ignore"
)

# Join into base_df
base_df = base_df.join(forecast_generation_df, how="left")

print("\n✅ Forecast wind + solar added to base_df")
print("base_df shape:", base_df.shape)

print("\nMissing values:")
display(base_df.isna().sum().to_frame("missing_count"))

print("\nOnly forecast missing:")
display(
    base_df[
        ["forecast_wind_generation_mw", "forecast_solar_generation_mw"]
    ].isna().sum().to_frame("missing_count")
)

display(base_df.head())
display(base_df.tail())

🌬️☀️ Fetching DK1 forecast wind + solar generation
Range: 2021-01-01 00:00:00+00:00 → 2026-05-28 20:00:00+00:00
Forecasts_Hour 2021-01: 2232 rows | offset 0
Forecasts_Hour 2021-02: 2016 rows | offset 0
Forecasts_Hour 2021-03: 2229 rows | offset 0
⚠️ 429 rate limit. Sleeping 20 seconds then retrying...
Forecasts_Hour 2021-04: 2160 rows | offset 0
Forecasts_Hour 2021-05: 2226 rows | offset 0
Forecasts_Hour 2021-06: 2160 rows | offset 0
⚠️ 429 rate limit. Sleeping 20 seconds then retrying...
Forecasts_Hour 2021-07: 2232 rows | offset 0
Forecasts_Hour 2021-08: 2232 rows | offset 0
Forecasts_Hour 2021-09: 2160 rows | offset 0
⚠️ 429 rate limit. Sleeping 20 seconds then retrying...
Forecasts_Hour 2021-10: 2232 rows | offset 0
Forecasts_Hour 2021-11: 2160 rows | offset 0
Forecasts_Hour 2021-12: 2232 rows | offset 0
⚠️ 429 rate limit. Sleeping 20 seconds then retrying...
Forecasts_Hour 2022-01: 2232 rows | offset 0
Forecasts_Hour 2022-02: 2016 rows | offset 0
Forecasts_Hour 2022-03: 2229 rows 

,count
ForecastType,
Solar,47232
Onshore Wind,47205
Offshore Wind,47187



✅ Forecast wind + solar added to base_df
base_df shape: (47373, 28)

Missing values:


,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0
temperature,0
hour,0
day_of_week,0
month,0
day_of_year,0
hour_sin,0



Only forecast missing:


,missing_count
forecast_wind_generation_mw,144
forecast_solar_generation_mw,142


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,...,co2_lag_2h,co2_lag_24h,co2_lag_168h,co2_rolling_3h,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,5.2,0,0.7,1,4,1,1,0.258819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,220.250008,0.00000
2021-01-01 01:00:00+00:00,196.000000,0.33246,4.8,0,1.0,2,4,1,1,0.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.458336,0.00000
2021-01-01 02:00:00+00:00,168.166667,0.31937,6.5,0,1.2,3,4,1,1,0.707107,...,190.833333,NaN,NaN,NaN,NaN,NaN,5.166667,NaN,131.875004,0.00000
2021-01-01 03:00:00+00:00,160.083333,0.30054,6.6,0,1.3,4,4,1,1,0.866025,...,196.000000,NaN,NaN,185.00,NaN,NaN,-27.833333,NaN,116.750000,0.00000
2021-01-01 04:00:00+00:00,161.083333,0.29913,5.6,0,1.2,5,4,1,1,0.965926,...,168.166667,NaN,NaN,174.75,NaN,NaN,-8.083333,NaN,100.958332,0.00625


,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,...,co2_lag_2h,co2_lag_24h,co2_lag_168h,co2_rolling_3h,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2026-05-28 16:00:00+00:00,89.500000,0.296251,12.6,486,18.8,18,3,5,148,-1.000000,...,83.250000,19.000000,85.750000,112.138889,150.222222,85.854167,-5.083333,63.833333,269.625000,1106.832886
2026-05-28 17:00:00+00:00,105.500000,0.756657,10.8,347,17.7,19,3,5,148,-0.965926,...,78.166667,28.750000,106.750000,83.638889,133.958333,88.791667,11.333333,70.500000,203.583328,625.209167
2026-05-28 18:00:00+00:00,121.416667,1.033087,10.4,210,16.5,20,3,5,148,-0.866025,...,89.500000,35.333333,105.500000,91.055556,118.916667,91.989583,16.000000,76.750000,149.333336,238.740005
2026-05-28 19:00:00+00:00,126.583333,1.626056,9.4,98,15.4,21,3,5,148,-0.707107,...,105.500000,51.833333,119.666667,105.472222,108.805556,95.576389,15.916667,86.083333,NaN,NaN
2026-05-28 20:00:00+00:00,132.000000,2.943719,8.6,19,14.0,22,3,5,148,-0.500000,...,121.416667,50.333333,121.750000,117.833333,100.736111,98.690972,5.166667,74.750000,NaN,NaN


In [73]:
base_df[base_df["forecast_wind_generation_mw"].isna()][
    ["forecast_wind_generation_mw", "forecast_solar_generation_mw"]]

,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,
2021-10-07 22:00:00+00:00,NaN,NaN
2024-04-12 22:00:00+00:00,NaN,NaN
2024-04-12 23:00:00+00:00,NaN,NaN
2024-04-13 00:00:00+00:00,NaN,NaN
2024-04-13 01:00:00+00:00,NaN,NaN
...,...,...
2025-11-24 11:00:00+00:00,NaN,NaN
2025-11-24 12:00:00+00:00,NaN,NaN
2025-11-24 13:00:00+00:00,NaN,NaN


In [74]:
a = base_df.copy()

In [75]:
# ============================================================
# STEP 8B — Re-fetch only missing forecast wind/solar hours
# Input: base_df
# Output: base_df with missing forecast values patched from API
# ============================================================

import requests
import pandas as pd
import numpy as np
import json
import time

forecast_cols = [
    "forecast_wind_generation_mw",
    "forecast_solar_generation_mw"
]

# 1. Find missing forecast timestamps
missing_mask = base_df[forecast_cols].isna().any(axis=1)
missing_times = base_df.index[missing_mask]

print("Missing forecast rows before:", len(missing_times))

if len(missing_times) == 0:
    print("✅ No missing forecast data.")
else:
    display(base_df.loc[missing_times, forecast_cols].head(20))

    # Fetch by unique date to reduce API calls
    missing_dates = sorted(set(missing_times.date))
    print("Unique missing dates:", len(missing_dates))
    print(missing_dates[:10], "..." if len(missing_dates) > 10 else "")

    def request_with_retry(url, params, max_retries=5):
        for attempt in range(max_retries):
            r = requests.get(url, params=params, timeout=60)

            if r.status_code == 200:
                return r

            if r.status_code == 429:
                sleep_time = 30 + attempt * 30
                print(f"⚠️ 429 rate limit. Sleeping {sleep_time} seconds...")
                time.sleep(sleep_time)
                continue

            print("⚠️ Request failed:", r.status_code)
            print(r.text[:300])
            return r

        return r

    patch_rows = []
    url = "https://api.energidataservice.dk/dataset/Forecasts_Hour"

    for d in missing_dates:
        day_start = pd.Timestamp(d, tz="UTC")
        day_end = day_start + pd.Timedelta(days=1)

        params = {
            "start": day_start.strftime("%Y-%m-%dT%H:%M"),
            "end": day_end.strftime("%Y-%m-%dT%H:%M"),
            "filter": json.dumps({"PriceArea": ["DK1"]}),
            "columns": "HourUTC,PriceArea,ForecastType,ForecastDayAhead",
            "sort": "HourUTC",
            "limit": 100000
        }

        r = request_with_retry(url, params)

        if r.status_code == 200:
            records = r.json().get("records", [])
            patch_rows.extend(records)
            print(f"{d}: fetched {len(records)} records")
        else:
            print(f"❌ Failed for {d}")

        time.sleep(1.0)

    patch_raw_df = pd.DataFrame(patch_rows)

    if patch_raw_df.empty:
        print("⚠️ No patch data returned from API.")
    else:
        patch_raw_df["timestamp_utc"] = pd.to_datetime(
            patch_raw_df["HourUTC"],
            utc=True,
            errors="coerce"
        )

        patch_raw_df["ForecastDayAhead"] = pd.to_numeric(
            patch_raw_df["ForecastDayAhead"],
            errors="coerce"
        )

        patch_raw_df["forecast_type_clean"] = (
            patch_raw_df["ForecastType"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

        wind_mask = patch_raw_df["forecast_type_clean"].str.contains("wind", na=False)
        solar_mask = patch_raw_df["forecast_type_clean"].str.contains("solar", na=False)

        patch_long_df = patch_raw_df[wind_mask | solar_mask].copy()

        patch_long_df["feature"] = np.where(
            patch_long_df["forecast_type_clean"].str.contains("solar", na=False),
            "forecast_solar_generation_mw",
            "forecast_wind_generation_mw"
        )

        patch_forecast_df = (
            patch_long_df
            .groupby(["timestamp_utc", "feature"], as_index=False)["ForecastDayAhead"]
            .sum()
            .pivot(index="timestamp_utc", columns="feature", values="ForecastDayAhead")
            .sort_index()
        )

        # Keep only base_df timestamps
        patch_forecast_df = patch_forecast_df.reindex(base_df.index)

        for col in forecast_cols:
            if col in patch_forecast_df.columns:
                fill_mask = base_df[col].isna() & patch_forecast_df[col].notna()
                base_df.loc[fill_mask, col] = patch_forecast_df.loc[fill_mask, col]
                print(f"Filled {col}:", fill_mask.sum())

        print("\nMissing forecast rows after:")
        display(base_df[forecast_cols].isna().sum().to_frame("missing_count"))

        display(base_df.loc[base_df[forecast_cols].isna().any(axis=1), forecast_cols].head(20))

Missing forecast rows before: 162


,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,
2021-10-07 22:00:00+00:00,NaN,NaN
2024-04-12 22:00:00+00:00,NaN,NaN
2024-04-12 23:00:00+00:00,NaN,NaN
2024-04-13 00:00:00+00:00,NaN,NaN
2024-04-13 01:00:00+00:00,NaN,NaN
2024-04-13 02:00:00+00:00,NaN,NaN
2024-04-13 03:00:00+00:00,NaN,NaN
2024-04-13 04:00:00+00:00,NaN,NaN
2024-04-13 05:00:00+00:00,NaN,NaN


Unique missing dates: 16
[datetime.date(2021, 10, 7), datetime.date(2024, 4, 12), datetime.date(2024, 4, 13), datetime.date(2024, 4, 14), datetime.date(2024, 4, 15), datetime.date(2025, 4, 9), datetime.date(2025, 4, 10), datetime.date(2025, 8, 29), datetime.date(2025, 8, 30), datetime.date(2025, 9, 22)] ...
2021-10-07: fetched 72 records
2024-04-12: fetched 72 records
2024-04-13: fetched 0 records
⚠️ 429 rate limit. Sleeping 30 seconds...
2024-04-14: fetched 0 records
2024-04-15: fetched 54 records
2025-04-09: fetched 72 records
⚠️ 429 rate limit. Sleeping 30 seconds...
2025-04-10: fetched 42 records
2025-08-29: fetched 72 records
2025-08-30: fetched 52 records
⚠️ 429 rate limit. Sleeping 30 seconds...
2025-09-22: fetched 72 records
2025-09-23: fetched 31 records
2025-11-21: fetched 72 records
⚠️ 429 rate limit. Sleeping 30 seconds...
2025-11-22: fetched 0 records
2025-11-23: fetched 0 records
2025-11-24: fetched 25 records
⚠️ 429 rate limit. Sleeping 30 seconds...
2026-05-28: fetched 

,missing_count
forecast_wind_generation_mw,142
forecast_solar_generation_mw,140


,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,
2021-10-07 22:00:00+00:00,NaN,NaN
2024-04-12 22:00:00+00:00,NaN,NaN
2024-04-12 23:00:00+00:00,NaN,NaN
2024-04-13 00:00:00+00:00,NaN,NaN
2024-04-13 01:00:00+00:00,NaN,NaN
2024-04-13 02:00:00+00:00,NaN,NaN
2024-04-13 03:00:00+00:00,NaN,NaN
2024-04-13 04:00:00+00:00,NaN,NaN
2024-04-13 05:00:00+00:00,NaN,NaN


In [77]:
# ============================================================
# STEP 8D — Fill missing wind/solar forecast using ENTSO-E token from .env
# Input: base_df with missing:
#   forecast_wind_generation_mw
#   forecast_solar_generation_mw
#
# Output: base_df patched using ENTSO-E where possible
# ============================================================

import os
import time
import requests
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

# ------------------------------------------------------------
# 1. Load ENTSO-E token from .env
# Your .env should contain:
# ENTSOE_TOKEN=your_token_here
# ------------------------------------------------------------

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    !pip install python-dotenv
    from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path)
    print("✅ .env loaded from:", dotenv_path)
else:
    load_dotenv()
    print("⚠️ .env file not found by find_dotenv(), trying default load_dotenv().")

ENTSOE_TOKEN = os.getenv("ENTSOE_TOKEN")

if not ENTSOE_TOKEN:
    raise ValueError("ENTSOE_TOKEN not found. Please check your .env file.")

print("✅ ENTSO-E token loaded")


# ------------------------------------------------------------
# 2. Config
# ------------------------------------------------------------

# DK1 bidding zone EIC code
DK1_EIC = "10YDK-1--------W"

forecast_cols = [
    "forecast_wind_generation_mw",
    "forecast_solar_generation_mw"
]

# ENTSO-E PSR type mapping
# B16 = Solar
# B18 = Wind Offshore
# B19 = Wind Onshore
PSR_TO_FEATURE = {
    "B16": "forecast_solar_generation_mw",
    "B18": "forecast_wind_generation_mw",
    "B19": "forecast_wind_generation_mw",
}


# ------------------------------------------------------------
# 3. Find missing forecast timestamps
# ------------------------------------------------------------

missing_mask = base_df[forecast_cols].isna().any(axis=1)
missing_times = base_df.index[missing_mask]

print("\nMissing forecast rows before ENTSO-E patch:", len(missing_times))

if len(missing_times) == 0:
    print("✅ No missing wind/solar forecast values.")
else:
    missing_dates = sorted(set(missing_times.normalize()))

    print("Unique missing UTC dates:", len(missing_dates))
    print("First missing dates:", [d.date() for d in missing_dates[:10]])


# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------

def parse_entsoe_resolution(resolution_text):
    """
    Convert ENTSO-E resolution text like PT60M, PT15M, PT1H to pandas Timedelta.
    """
    resolution_text = resolution_text.strip()

    if resolution_text.startswith("PT") and resolution_text.endswith("M"):
        minutes = int(resolution_text.replace("PT", "").replace("M", ""))
        return pd.Timedelta(minutes=minutes)

    if resolution_text.startswith("PT") and resolution_text.endswith("H"):
        hours = int(resolution_text.replace("PT", "").replace("H", ""))
        return pd.Timedelta(hours=hours)

    raise ValueError(f"Unsupported ENTSO-E resolution: {resolution_text}")


def request_entsoe_with_retry(params, max_retries=5):
    """
    Request ENTSO-E API with retry for rate limits.
    """
    url = "https://web-api.tp.entsoe.eu/api"

    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=90)

        if response.status_code == 200:
            return response

        if response.status_code == 429:
            sleep_time = 30 + attempt * 30
            print(f"⚠️ ENTSO-E 429 rate limit. Sleeping {sleep_time}s...")
            time.sleep(sleep_time)
            continue

        print("⚠️ ENTSO-E request failed")
        print("Status:", response.status_code)
        print(response.text[:700])
        return response

    return response


def fetch_entsoe_wind_solar_for_day(day_start_utc):
    """
    Fetch ENTSO-E wind/solar day-ahead forecast for one UTC day.

    documentType A69 = Wind and solar forecast
    processType A01 = Day ahead
    in_Domain = DK1 bidding zone
    """

    day_start_utc = pd.Timestamp(day_start_utc).tz_convert("UTC")
    day_end_utc = day_start_utc + pd.Timedelta(days=1)

    params = {
        "securityToken": ENTSOE_TOKEN,
        "documentType": "A69",
        "processType": "A01",
        "in_Domain": DK1_EIC,
        "periodStart": day_start_utc.strftime("%Y%m%d%H%M"),
        "periodEnd": day_end_utc.strftime("%Y%m%d%H%M"),
    }

    response = request_entsoe_with_retry(params)

    if response.status_code != 200:
        return pd.DataFrame()

    try:
        root = ET.fromstring(response.content)
    except Exception as e:
        print("❌ XML parse error:", e)
        print(response.text[:700])
        return pd.DataFrame()

    # Detect namespace
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}")[0].replace("{", "")
        ns = {"ns": ns_uri}
        prefix = "ns:"
    else:
        ns = {}
        prefix = ""

    rows = []

    for ts in root.findall(f".//{prefix}TimeSeries", ns):

        # Find PSR type
        psr_node = ts.find(f".//{prefix}MktPSRType/{prefix}psrType", ns)

        if psr_node is None:
            psr_node = ts.find(f".//{prefix}mktPSRType/{prefix}psrType", ns)

        if psr_node is None or psr_node.text is None:
            continue

        psr_type = psr_node.text.strip()

        if psr_type not in PSR_TO_FEATURE:
            continue

        feature = PSR_TO_FEATURE[psr_type]

        for period in ts.findall(f".//{prefix}Period", ns):

            start_node = period.find(f".//{prefix}timeInterval/{prefix}start", ns)
            resolution_node = period.find(f".//{prefix}resolution", ns)

            if start_node is None or resolution_node is None:
                continue

            period_start = pd.to_datetime(start_node.text, utc=True)
            step = parse_entsoe_resolution(resolution_node.text)

            for point in period.findall(f".//{prefix}Point", ns):

                pos_node = point.find(f"{prefix}position", ns)
                qty_node = point.find(f"{prefix}quantity", ns)

                if pos_node is None or qty_node is None:
                    continue

                try:
                    position = int(pos_node.text)
                    quantity = float(qty_node.text)
                except:
                    continue

                timestamp = period_start + (position - 1) * step

                rows.append({
                    "timestamp_utc": timestamp,
                    "feature": feature,
                    "value": quantity
                })

    if len(rows) == 0:
        return pd.DataFrame()

    temp = pd.DataFrame(rows)

    # First sum wind onshore + offshore if both exist at same timestamp
    temp = (
        temp
        .groupby(["timestamp_utc", "feature"], as_index=False)["value"]
        .sum()
    )

    # If ENTSO-E returns sub-hourly values, convert to hourly mean
    temp = (
        temp
        .set_index("timestamp_utc")
        .groupby("feature")["value"]
        .resample("h")
        .mean()
        .reset_index()
    )

    # Pivot to wide format
    temp = (
        temp
        .pivot(index="timestamp_utc", columns="feature", values="value")
        .sort_index()
    )

    return temp


# ------------------------------------------------------------
# 5. Fetch only missing dates from ENTSO-E
# ------------------------------------------------------------

if len(missing_times) > 0:

    entsoe_patch_list = []

    for day in missing_dates:
        print("\nFetching ENTSO-E:", day.date())

        day_df = fetch_entsoe_wind_solar_for_day(day)

        if day_df.empty:
            print("  ⚠️ No ENTSO-E data returned")
        else:
            entsoe_patch_list.append(day_df)
            print("  ✅ rows:", len(day_df))
            print("  columns:", list(day_df.columns))

        time.sleep(1.5)

    # --------------------------------------------------------
    # 6. Patch base_df
    # --------------------------------------------------------

    if len(entsoe_patch_list) == 0:
        print("\n⚠️ ENTSO-E returned no usable patch data.")
    else:
        entsoe_patch_df = pd.concat(entsoe_patch_list).sort_index()

        # Remove possible duplicate timestamps
        entsoe_patch_df = entsoe_patch_df.groupby(entsoe_patch_df.index).mean()

        # Align exactly to base_df index
        entsoe_patch_df = entsoe_patch_df.reindex(base_df.index)

        print("\nENTSO-E patch dataframe:")
        print(entsoe_patch_df.shape)
        display(entsoe_patch_df.head())
        display(entsoe_patch_df.tail())

        for col in forecast_cols:
            if col in entsoe_patch_df.columns:
                fill_mask = base_df[col].isna() & entsoe_patch_df[col].notna()

                base_df.loc[fill_mask, col] = entsoe_patch_df.loc[fill_mask, col]

                print(f"Filled {col} from ENTSO-E:", fill_mask.sum())
            else:
                print(f"⚠️ {col} not found in ENTSO-E response")

        print("\nMissing forecast values after ENTSO-E patch:")
        display(base_df[forecast_cols].isna().sum().to_frame("missing_count"))

        print("\nRemaining missing rows preview:")
        display(
            base_df.loc[
                base_df[forecast_cols].isna().any(axis=1),
                forecast_cols
            ].head(30)
        )

✅ .env loaded from: /Users/praful/Documents/Aalborg_University/2nd_Sem/Semester_Project/.env
✅ ENTSO-E token loaded

Missing forecast rows before ENTSO-E patch: 160
Unique missing UTC dates: 15
First missing dates: [datetime.date(2021, 10, 7), datetime.date(2024, 4, 12), datetime.date(2024, 4, 13), datetime.date(2024, 4, 14), datetime.date(2024, 4, 15), datetime.date(2025, 4, 9), datetime.date(2025, 4, 10), datetime.date(2025, 8, 29), datetime.date(2025, 8, 30), datetime.date(2025, 9, 22)]

Fetching ENTSO-E: 2021-10-07
  ✅ rows: 22
  columns: ['forecast_solar_generation_mw', 'forecast_wind_generation_mw']

Fetching ENTSO-E: 2024-04-12
  ✅ rows: 24
  columns: ['forecast_solar_generation_mw', 'forecast_wind_generation_mw']

Fetching ENTSO-E: 2024-04-13
  ✅ rows: 24
  columns: ['forecast_solar_generation_mw', 'forecast_wind_generation_mw']

Fetching ENTSO-E: 2024-04-14
  ✅ rows: 24
  columns: ['forecast_solar_generation_mw', 'forecast_wind_generation_mw']

Fetching ENTSO-E: 2024-04-15
  ✅

feature,forecast_solar_generation_mw,forecast_wind_generation_mw
timestamp_utc,,
2021-01-01 00:00:00+00:00,NaN,NaN
2021-01-01 01:00:00+00:00,NaN,NaN
2021-01-01 02:00:00+00:00,NaN,NaN
2021-01-01 03:00:00+00:00,NaN,NaN
2021-01-01 04:00:00+00:00,NaN,NaN


feature,forecast_solar_generation_mw,forecast_wind_generation_mw
timestamp_utc,,
2026-05-28 16:00:00+00:00,NaN,NaN
2026-05-28 17:00:00+00:00,NaN,NaN
2026-05-28 18:00:00+00:00,NaN,NaN
2026-05-28 19:00:00+00:00,NaN,NaN
2026-05-28 20:00:00+00:00,NaN,NaN


Filled forecast_wind_generation_mw from ENTSO-E: 130
Filled forecast_solar_generation_mw from ENTSO-E: 84

Missing forecast values after ENTSO-E patch:


,missing_count
forecast_wind_generation_mw,12
forecast_solar_generation_mw,56



Remaining missing rows preview:


,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,
2021-10-07 22:00:00+00:00,NaN,NaN
2024-04-12 22:00:00+00:00,1820.2900,NaN
2024-04-12 23:00:00+00:00,1820.4200,NaN
2024-04-13 01:00:00+00:00,1257.0000,NaN
2024-04-13 02:00:00+00:00,2058.0000,NaN
2024-04-13 21:00:00+00:00,2832.2100,NaN
2024-04-13 22:00:00+00:00,3211.5000,NaN
2024-04-13 23:00:00+00:00,3309.2900,NaN
2024-04-14 01:00:00+00:00,3367.0900,NaN


In [78]:
# ============================================================
# STEP 8E — Final patch for remaining wind/solar missing values
# ============================================================

forecast_cols = [
    "forecast_wind_generation_mw",
    "forecast_solar_generation_mw"
]

print("Before final patch:")
display(base_df[forecast_cols].isna().sum().to_frame("missing_count"))

# 1. Solar: if weather solar radiation is zero, solar generation forecast should be zero
solar_zero_mask = (
    base_df["forecast_solar_generation_mw"].isna() &
    (base_df["solar_radiation"] == 0)
)

base_df.loc[solar_zero_mask, "forecast_solar_generation_mw"] = 0

print("Solar missing set to 0 using solar_radiation == 0:", solar_zero_mask.sum())

# 2. Interpolate small remaining gaps
for col in forecast_cols:
    base_df[col] = base_df[col].interpolate(
        method="time",
        limit=6,
        limit_direction="both"
    )

# 3. Fill larger gaps using same hour previous week
for col in forecast_cols:
    base_df[col] = base_df[col].fillna(base_df[col].shift(168))

# 4. Final fallback: same hour next week
for col in forecast_cols:
    base_df[col] = base_df[col].fillna(base_df[col].shift(-168))

print("\nAfter final patch:")
display(base_df[forecast_cols].isna().sum().to_frame("missing_count"))

print("\nRemaining missing rows:")
display(
    base_df.loc[
        base_df[forecast_cols].isna().any(axis=1),
        forecast_cols
    ]
)

display(base_df[forecast_cols].describe())

Before final patch:


,missing_count
forecast_wind_generation_mw,12
forecast_solar_generation_mw,56


Solar missing set to 0 using solar_radiation == 0: 47

After final patch:


,missing_count
forecast_wind_generation_mw,0
forecast_solar_generation_mw,0



Remaining missing rows:


,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,


,forecast_wind_generation_mw,forecast_solar_generation_mw
count,47373.000000,47373.000000
mean,1461.578588,230.905263
std,1001.819915,401.377522
min,0.000000,0.000000
25%,606.666688,0.000000
50%,1272.333313,5.440000
75%,2180.499939,303.537079
max,4477.083374,2359.466553


In [79]:
base_df.to_csv("base_df.csv", index=True)

In [80]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 47373 entries, 2021-01-01 00:00:00+00:00 to 2026-05-28 20:00:00+00:00
Freq: h
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   co2_emissions_g_kwh           47373 non-null  float64
 1   spot_price_dkk_kwh            47373 non-null  float64
 2   wind_speed                    47373 non-null  float64
 3   solar_radiation               47373 non-null  int64  
 4   temperature                   47373 non-null  float64
 5   hour                          47373 non-null  int32  
 6   day_of_week                   47373 non-null  int32  
 7   month                         47373 non-null  int32  
 8   day_of_year                   47373 non-null  int32  
 9   hour_sin                      47373 non-null  float64
 10  hour_cos                      47373 non-null  float64
 11  month_sin                     47373 non-null  float64
 12  month

In [81]:
# ============================================================
# STEP 8E — Final patch for remaining wind/solar missing values
# ============================================================

forecast_cols = [
    "forecast_wind_generation_mw",
    "forecast_solar_generation_mw"
]

print("Before final patch:")
display(base_df[forecast_cols].isna().sum().to_frame("missing_count"))

# 1. Solar: if solar_radiation is 0, missing solar forecast can be set to 0
solar_zero_mask = (
    base_df["forecast_solar_generation_mw"].isna() &
    (base_df["solar_radiation"] == 0)
)

base_df.loc[solar_zero_mask, "forecast_solar_generation_mw"] = 0

print("\nSolar missing set to 0 because solar_radiation == 0:")
print(solar_zero_mask.sum())

# 2. Interpolate small gaps up to 6 hours
for col in forecast_cols:
    base_df[col] = base_df[col].interpolate(
        method="time",
        limit=6,
        limit_direction="both"
    )

# 3. Fill remaining larger gaps using same hour previous week
for col in forecast_cols:
    base_df[col] = base_df[col].fillna(base_df[col].shift(168))

# 4. Final fallback: same hour next week
for col in forecast_cols:
    base_df[col] = base_df[col].fillna(base_df[col].shift(-168))

print("\nAfter final patch:")
display(base_df[forecast_cols].isna().sum().to_frame("missing_count"))

print("\nSummary after patch:")
display(base_df[forecast_cols].describe())

print("\nRemaining missing rows, if any:")
display(
    base_df.loc[
        base_df[forecast_cols].isna().any(axis=1),
        forecast_cols
    ]
)

Before final patch:


,missing_count
forecast_wind_generation_mw,0
forecast_solar_generation_mw,0



Solar missing set to 0 because solar_radiation == 0:
0

After final patch:


,missing_count
forecast_wind_generation_mw,0
forecast_solar_generation_mw,0



Summary after patch:


,forecast_wind_generation_mw,forecast_solar_generation_mw
count,47373.000000,47373.000000
mean,1461.578588,230.905263
std,1001.819915,401.377522
min,0.000000,0.000000
25%,606.666688,0.000000
50%,1272.333313,5.440000
75%,2180.499939,303.537079
max,4477.083374,2359.466553



Remaining missing rows, if any:


,forecast_wind_generation_mw,forecast_solar_generation_mw
timestamp_utc,,


In [82]:
# ============================================================
# STEP 9 — Add forecast_load_mw from ENTSO-E
# Input: base_df
# Output: base_df with forecast_load_mw
# ============================================================

import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    !pip install python-dotenv
    from dotenv import load_dotenv, find_dotenv

# ------------------------------------------------------------
# 1. Load ENTSO-E token
# .env should contain:
# ENTSOE_TOKEN=your_token_here
# ------------------------------------------------------------

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path)
    print("✅ .env loaded from:", dotenv_path)
else:
    load_dotenv()
    print("⚠️ .env not found by find_dotenv(), trying default load_dotenv().")

ENTSOE_TOKEN = os.getenv("ENTSOE_TOKEN")

if not ENTSOE_TOKEN:
    raise ValueError("ENTSOE_TOKEN not found. Check your .env file.")

print("✅ ENTSO-E token loaded")


# ------------------------------------------------------------
# 2. Config
# ------------------------------------------------------------

DK1_EIC = "10YDK-1--------W"

start_utc = base_df.index.min()
end_utc = base_df.index.max()

print("\n📦 Fetching DK1 forecast load from ENTSO-E")
print("Range:", start_utc, "→", end_utc)


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def parse_entsoe_resolution(resolution_text):
    resolution_text = resolution_text.strip()

    if resolution_text.startswith("PT") and resolution_text.endswith("M"):
        minutes = int(resolution_text.replace("PT", "").replace("M", ""))
        return pd.Timedelta(minutes=minutes)

    if resolution_text.startswith("PT") and resolution_text.endswith("H"):
        hours = int(resolution_text.replace("PT", "").replace("H", ""))
        return pd.Timedelta(hours=hours)

    raise ValueError(f"Unsupported resolution: {resolution_text}")


def request_entsoe_with_retry(params, max_retries=5):
    url = "https://web-api.tp.entsoe.eu/api"

    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=90)

        if response.status_code == 200:
            return response

        if response.status_code == 429:
            sleep_time = 30 + attempt * 30
            print(f"⚠️ ENTSO-E 429 rate limit. Sleeping {sleep_time}s...")
            time.sleep(sleep_time)
            continue

        print("⚠️ ENTSO-E request failed")
        print("Status:", response.status_code)
        print(response.text[:700])
        return response

    return response


def parse_entsoe_load_xml(xml_content):
    root = ET.fromstring(xml_content)

    # Detect namespace
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}")[0].replace("{", "")
        ns = {"ns": ns_uri}
        prefix = "ns:"
    else:
        ns = {}
        prefix = ""

    rows = []

    for ts in root.findall(f".//{prefix}TimeSeries", ns):
        for period in ts.findall(f".//{prefix}Period", ns):

            start_node = period.find(f".//{prefix}timeInterval/{prefix}start", ns)
            resolution_node = period.find(f".//{prefix}resolution", ns)

            if start_node is None or resolution_node is None:
                continue

            period_start = pd.to_datetime(start_node.text, utc=True)
            step = parse_entsoe_resolution(resolution_node.text)

            for point in period.findall(f".//{prefix}Point", ns):
                pos_node = point.find(f"{prefix}position", ns)
                qty_node = point.find(f"{prefix}quantity", ns)

                if pos_node is None or qty_node is None:
                    continue

                try:
                    position = int(pos_node.text)
                    quantity = float(qty_node.text)
                except:
                    continue

                timestamp = period_start + (position - 1) * step

                rows.append({
                    "timestamp_utc": timestamp,
                    "forecast_load_mw": quantity
                })

    if len(rows) == 0:
        return pd.DataFrame()

    temp = pd.DataFrame(rows)

    temp["timestamp_utc"] = pd.to_datetime(temp["timestamp_utc"], utc=True)
    temp["forecast_load_mw"] = pd.to_numeric(temp["forecast_load_mw"], errors="coerce")

    # If ENTSO-E gives sub-hourly values, convert to hourly mean
    temp = (
        temp
        .set_index("timestamp_utc")
        .resample("h")["forecast_load_mw"]
        .mean()
        .to_frame()
        .sort_index()
    )

    return temp


def fetch_entsoe_load_forecast_chunk(chunk_start_utc, chunk_end_utc):
    """
    Fetch ENTSO-E day-ahead load forecast for one chunk.
    documentType A65 = System total load
    processType A01 = Day ahead
    """

    params = {
        "securityToken": ENTSOE_TOKEN,
        "documentType": "A65",
        "processType": "A01",
        "outBiddingZone_Domain": DK1_EIC,
        "periodStart": chunk_start_utc.strftime("%Y%m%d%H%M"),
        "periodEnd": chunk_end_utc.strftime("%Y%m%d%H%M"),
    }

    response = request_entsoe_with_retry(params)

    if response.status_code != 200:
        return pd.DataFrame()

    try:
        return parse_entsoe_load_xml(response.content)
    except Exception as e:
        print("❌ XML parse error:", e)
        print(response.text[:700])
        return pd.DataFrame()


# ------------------------------------------------------------
# 4. Fetch month by month
# ------------------------------------------------------------

load_parts = []

current = start_utc.replace(day=1, hour=0, minute=0, second=0, microsecond=0)

while current <= end_utc:
    next_month = current + pd.DateOffset(months=1)

    chunk_start = max(current, start_utc)
    chunk_end = min(next_month, end_utc + pd.Timedelta(hours=1))

    print(f"\nFetching load forecast: {chunk_start} → {chunk_end}")

    chunk_df = fetch_entsoe_load_forecast_chunk(chunk_start, chunk_end)

    if chunk_df.empty:
        print("  ⚠️ No data returned")
    else:
        load_parts.append(chunk_df)
        print("  ✅ rows:", len(chunk_df))

    current = next_month
    time.sleep(1.5)


# ------------------------------------------------------------
# 5. Merge into base_df
# ------------------------------------------------------------

if len(load_parts) == 0:
    raise ValueError("No forecast load data fetched from ENTSO-E.")

forecast_load_df = pd.concat(load_parts).sort_index()

# Remove duplicates if any
forecast_load_df = forecast_load_df.groupby(forecast_load_df.index).mean()

# Align exactly to base_df
forecast_load_df = forecast_load_df.reindex(base_df.index)

# Remove old column if rerunning
base_df = base_df.drop(columns=["forecast_load_mw"], errors="ignore")

# Join
base_df = base_df.join(forecast_load_df, how="left")

print("\n✅ forecast_load_mw added to base_df")
print("base_df shape:", base_df.shape)

print("\nMissing values:")
display(base_df.isna().sum().to_frame("missing_count"))

print("\nforecast_load_mw missing only:")
display(base_df[["forecast_load_mw"]].isna().sum().to_frame("missing_count"))

display(base_df[["forecast_load_mw"]].head())
display(base_df[["forecast_load_mw"]].tail())

✅ .env loaded from: /Users/praful/Documents/Aalborg_University/2nd_Sem/Semester_Project/.env
✅ ENTSO-E token loaded

📦 Fetching DK1 forecast load from ENTSO-E
Range: 2021-01-01 00:00:00+00:00 → 2026-05-28 20:00:00+00:00

Fetching load forecast: 2021-01-01 00:00:00+00:00 → 2021-02-01 00:00:00+00:00
  ✅ rows: 744

Fetching load forecast: 2021-02-01 00:00:00+00:00 → 2021-03-01 00:00:00+00:00
  ✅ rows: 672

Fetching load forecast: 2021-03-01 00:00:00+00:00 → 2021-04-01 00:00:00+00:00
  ✅ rows: 744

Fetching load forecast: 2021-04-01 00:00:00+00:00 → 2021-05-01 00:00:00+00:00
  ✅ rows: 720

Fetching load forecast: 2021-05-01 00:00:00+00:00 → 2021-06-01 00:00:00+00:00
  ✅ rows: 744

Fetching load forecast: 2021-06-01 00:00:00+00:00 → 2021-07-01 00:00:00+00:00
  ✅ rows: 720

Fetching load forecast: 2021-07-01 00:00:00+00:00 → 2021-08-01 00:00:00+00:00
  ✅ rows: 744

Fetching load forecast: 2021-08-01 00:00:00+00:00 → 2021-09-01 00:00:00+00:00
  ✅ rows: 744

Fetching load forecast: 2021-09-01 

,missing_count
co2_emissions_g_kwh,0
spot_price_dkk_kwh,0
wind_speed,0
solar_radiation,0
temperature,0
hour,0
day_of_week,0
month,0
day_of_year,0
hour_sin,0



forecast_load_mw missing only:


,missing_count
forecast_load_mw,211


,forecast_load_mw
timestamp_utc,
2021-01-01 00:00:00+00:00,2018.0
2021-01-01 01:00:00+00:00,1947.0
2021-01-01 02:00:00+00:00,1879.0
2021-01-01 03:00:00+00:00,1846.0
2021-01-01 04:00:00+00:00,1857.0


,forecast_load_mw
timestamp_utc,
2026-05-28 16:00:00+00:00,2595.0
2026-05-28 17:00:00+00:00,2538.0
2026-05-28 18:00:00+00:00,2421.0
2026-05-28 19:00:00+00:00,2414.0
2026-05-28 20:00:00+00:00,2413.0


In [83]:
# ============================================================
# STEP 9B — Patch remaining missing forecast_load_mw
# ============================================================

print("Before patch:")
display(base_df[["forecast_load_mw"]].isna().sum().to_frame("missing_count"))

# 1. Interpolate small gaps up to 6 hours
base_df["forecast_load_mw"] = base_df["forecast_load_mw"].interpolate(
    method="time",
    limit=6,
    limit_direction="both"
)

# 2. Fill larger gaps using same hour previous week
base_df["forecast_load_mw"] = base_df["forecast_load_mw"].fillna(
    base_df["forecast_load_mw"].shift(168)
)

# 3. Final fallback: same hour next week
base_df["forecast_load_mw"] = base_df["forecast_load_mw"].fillna(
    base_df["forecast_load_mw"].shift(-168)
)

print("\nAfter patch:")
display(base_df[["forecast_load_mw"]].isna().sum().to_frame("missing_count"))

print("\nSummary:")
display(base_df[["forecast_load_mw"]].describe())

print("\nRemaining missing rows:")
display(base_df.loc[base_df["forecast_load_mw"].isna(), ["forecast_load_mw"]])

Before patch:


,missing_count
forecast_load_mw,211



After patch:


,missing_count
forecast_load_mw,0



Summary:


,forecast_load_mw
count,47373.000000
mean,2570.397857
std,465.753907
min,1171.000000
25%,2220.000000
50%,2569.000000
75%,2907.000000
max,4308.000000



Remaining missing rows:


,forecast_load_mw
timestamp_utc,


In [84]:
base_df.to_csv("base_df_with_temperature.csv", index=True)

In [85]:
base_df.columns

Index(['co2_emissions_g_kwh', 'spot_price_dkk_kwh', 'wind_speed',
       'solar_radiation', 'temperature', 'hour', 'day_of_week', 'month',
       'day_of_year', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
       'day_of_year_sin', 'day_of_year_cos', 'is_weekend', 'is_holiday',
       'co2_lag_1h', 'co2_lag_2h', 'co2_lag_24h', 'co2_lag_168h',
       'co2_rolling_3h', 'co2_rolling_6h', 'co2_rolling_24h', 'co2_diff_1h',
       'co2_diff_24h', 'forecast_wind_generation_mw',
       'forecast_solar_generation_mw', 'forecast_load_mw'],
      dtype='object')

In [86]:
base_df.head()

,co2_emissions_g_kwh,spot_price_dkk_kwh,wind_speed,solar_radiation,temperature,hour,day_of_week,month,day_of_year,hour_sin,...,co2_lag_24h,co2_lag_168h,co2_rolling_3h,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h,forecast_wind_generation_mw,forecast_solar_generation_mw,forecast_load_mw
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,5.2,0,0.7,1,4,1,1,0.258819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,220.250008,0.00000,2018.0
2021-01-01 01:00:00+00:00,196.000000,0.33246,4.8,0,1.0,2,4,1,1,0.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.458336,0.00000,1947.0
2021-01-01 02:00:00+00:00,168.166667,0.31937,6.5,0,1.2,3,4,1,1,0.707107,...,NaN,NaN,NaN,NaN,NaN,5.166667,NaN,131.875004,0.00000,1879.0
2021-01-01 03:00:00+00:00,160.083333,0.30054,6.6,0,1.3,4,4,1,1,0.866025,...,NaN,NaN,185.00,NaN,NaN,-27.833333,NaN,116.750000,0.00000,1846.0
2021-01-01 04:00:00+00:00,161.083333,0.29913,5.6,0,1.2,5,4,1,1,0.965926,...,NaN,NaN,174.75,NaN,NaN,-8.083333,NaN,100.958332,0.00625,1857.0


In [2]:
base_df=pd.read_csv("base_df_with_temperature.csv", index_col=0, parse_dates=True)

In [3]:
# ============================================================
# STEP 4 — Add FORECASTED weather from Open-Meteo
# Input: base_df with timestamp_utc index
# Output: base_df with forecast-style:
#         wind_speed, solar_radiation, temperature
# Location: Aalborg, DK1
# ============================================================

import requests
import pandas as pd
import numpy as np
import time

# Aalborg coordinates
LATITUDE = 57.0488
LONGITUDE = 9.9217

# Open-Meteo Historical Forecast API
# This gives past weather forecast/model data, not archive/reanalysis weather
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

# Make sure base_df index is UTC datetime
base_df = base_df.copy()
base_df.index = pd.to_datetime(base_df.index, utc=True)
base_df = base_df.sort_index()

start_date = base_df.index.min().date()
end_date = base_df.index.max().date()

print("🌦️ Fetching FORECASTED weather from Open-Meteo Historical Forecast API")
print("Range:", start_date, "→", end_date)

# Fetch month by month to avoid very large API requests
date_ranges = pd.date_range(start=start_date, end=end_date, freq="MS")

# Make sure first month start is included
if date_ranges[0].date() > start_date:
    date_ranges = pd.DatetimeIndex([pd.Timestamp(start_date)]).append(date_ranges)

weather_parts = []

for i, chunk_start in enumerate(date_ranges):
    chunk_start = chunk_start.date()

    if i + 1 < len(date_ranges):
        chunk_end = (date_ranges[i + 1] - pd.Timedelta(days=1)).date()
    else:
        chunk_end = end_date

    # Do not go beyond final date
    chunk_end = min(chunk_end, end_date)

    print(f"📡 Fetching {chunk_start} → {chunk_end}")

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": str(chunk_start),
        "end_date": str(chunk_end),
        "hourly": "temperature_2m,wind_speed_10m,shortwave_radiation",
        "timezone": "UTC",
        "wind_speed_unit": "ms"   # important: get wind speed in m/s, not km/h
    }

    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    data = response.json()

    if "hourly" not in data:
        raise ValueError(f"No hourly data returned for {chunk_start} → {chunk_end}: {data}")

    temp_df = pd.DataFrame({
        "timestamp_utc": pd.to_datetime(data["hourly"]["time"], utc=True),
        "temperature": data["hourly"]["temperature_2m"],
        "wind_speed": data["hourly"]["wind_speed_10m"],
        "solar_radiation": data["hourly"]["shortwave_radiation"],
    })

    weather_parts.append(temp_df)

    # polite pause for API
    time.sleep(0.5)

forecast_weather_df = pd.concat(weather_parts, ignore_index=True)

forecast_weather_df = (
    forecast_weather_df
    .drop_duplicates(subset=["timestamp_utc"])
    .set_index("timestamp_utc")
    .sort_index()
)

# Convert numeric columns safely
for col in ["temperature", "wind_speed", "solar_radiation"]:
    forecast_weather_df[col] = pd.to_numeric(forecast_weather_df[col], errors="coerce")

print("\n✅ Forecasted weather downloaded")
print("forecast_weather_df shape:", forecast_weather_df.shape)

# Remove old actual weather columns if they already exist
for col in ["temperature", "wind_speed", "solar_radiation"]:
    if col in base_df.columns:
        base_df = base_df.drop(columns=[col])

# Join forecasted weather into base_df
base_df = base_df.join(forecast_weather_df, how="left")

print("\n✅ Forecasted temperature, wind_speed, and solar_radiation added to base_df")
print("base_df shape:", base_df.shape)

print("\nMissing values:")
display(base_df[["temperature", "wind_speed", "solar_radiation"]].isna().sum().to_frame("missing_count"))

display(base_df.head())
display(base_df.tail())

🌦️ Fetching FORECASTED weather from Open-Meteo Historical Forecast API
Range: 2021-01-01 → 2026-05-28
📡 Fetching 2021-01-01 → 2021-01-31
📡 Fetching 2021-02-01 → 2021-02-28
📡 Fetching 2021-03-01 → 2021-03-31
📡 Fetching 2021-04-01 → 2021-04-30
📡 Fetching 2021-05-01 → 2021-05-31
📡 Fetching 2021-06-01 → 2021-06-30
📡 Fetching 2021-07-01 → 2021-07-31
📡 Fetching 2021-08-01 → 2021-08-31
📡 Fetching 2021-09-01 → 2021-09-30
📡 Fetching 2021-10-01 → 2021-10-31
📡 Fetching 2021-11-01 → 2021-11-30
📡 Fetching 2021-12-01 → 2021-12-31
📡 Fetching 2022-01-01 → 2022-01-31
📡 Fetching 2022-02-01 → 2022-02-28
📡 Fetching 2022-03-01 → 2022-03-31
📡 Fetching 2022-04-01 → 2022-04-30
📡 Fetching 2022-05-01 → 2022-05-31
📡 Fetching 2022-06-01 → 2022-06-30
📡 Fetching 2022-07-01 → 2022-07-31
📡 Fetching 2022-08-01 → 2022-08-31
📡 Fetching 2022-09-01 → 2022-09-30
📡 Fetching 2022-10-01 → 2022-10-31
📡 Fetching 2022-11-01 → 2022-11-30
📡 Fetching 2022-12-01 → 2022-12-31
📡 Fetching 2023-01-01 → 2023-01-31
📡 Fetching 2023-02-01 →

,missing_count
temperature,0
wind_speed,0
solar_radiation,0


,co2_emissions_g_kwh,spot_price_dkk_kwh,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,month_sin,month_cos,...,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h,forecast_wind_generation_mw,forecast_solar_generation_mw,forecast_load_mw,temperature,wind_speed,solar_radiation
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00+00:00,190.833333,0.35858,1,4,1,1,0.258819,0.965926,0.5,0.866025,...,NaN,NaN,NaN,NaN,220.250008,0.00000,2018.0,0.7,1.92,0.0
2021-01-01 01:00:00+00:00,196.000000,0.33246,2,4,1,1,0.500000,0.866025,0.5,0.866025,...,NaN,NaN,NaN,NaN,170.458336,0.00000,1947.0,1.0,1.81,0.0
2021-01-01 02:00:00+00:00,168.166667,0.31937,3,4,1,1,0.707107,0.707107,0.5,0.866025,...,NaN,NaN,5.166667,NaN,131.875004,0.00000,1879.0,1.2,1.82,0.0
2021-01-01 03:00:00+00:00,160.083333,0.30054,4,4,1,1,0.866025,0.500000,0.5,0.866025,...,NaN,NaN,-27.833333,NaN,116.750000,0.00000,1846.0,1.3,1.73,0.0
2021-01-01 04:00:00+00:00,161.083333,0.29913,5,4,1,1,0.965926,0.258819,0.5,0.866025,...,NaN,NaN,-8.083333,NaN,100.958332,0.00625,1857.0,1.2,1.43,0.0


,co2_emissions_g_kwh,spot_price_dkk_kwh,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,month_sin,month_cos,...,co2_rolling_6h,co2_rolling_24h,co2_diff_1h,co2_diff_24h,forecast_wind_generation_mw,forecast_solar_generation_mw,forecast_load_mw,temperature,wind_speed,solar_radiation
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2026-05-28 16:00:00+00:00,89.500000,0.296251,18,3,5,148,-1.000000,-1.836970e-16,0.5,-0.866025,...,150.222222,85.854167,-5.083333,63.833333,269.625000,1106.832886,2595.0,21.8,3.5,486.0
2026-05-28 17:00:00+00:00,105.500000,0.756657,19,3,5,148,-0.965926,2.588190e-01,0.5,-0.866025,...,133.958333,88.791667,11.333333,70.500000,203.583328,625.209167,2538.0,21.3,3.0,347.0
2026-05-28 18:00:00+00:00,121.416667,1.033087,20,3,5,148,-0.866025,5.000000e-01,0.5,-0.866025,...,118.916667,91.989583,16.000000,76.750000,149.333336,238.740005,2421.0,20.2,2.9,210.0
2026-05-28 19:00:00+00:00,126.583333,1.626056,21,3,5,148,-0.707107,7.071068e-01,0.5,-0.866025,...,108.805556,95.576389,15.916667,86.083333,120.750004,25.858334,2414.0,18.3,2.6,98.0
2026-05-28 20:00:00+00:00,132.000000,2.943719,22,3,5,148,-0.500000,8.660254e-01,0.5,-0.866025,...,100.736111,98.690972,5.166667,74.750000,123.541670,0.000000,2413.0,16.6,2.4,19.0


In [5]:
base_df.to_csv("base_df_with_temperature.csv", index=True)